# 📕 Módulo 04 - Notebook 04: Depuración de Maestros de Clientes

## 🔍 Deduplicación Fuzzy y Validación de Datos

**Libro:** Saliendo de lo Pandito v4  
**Módulo:** 04 - Limpieza y Preparación de Datos  
**Duración estimada:** 55 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos

✅ **Validar** datos empresariales (CUIT, email, teléfono)  
✅ **Detectar** duplicados fuzzy (similitud de strings)  
✅ **Consolidar** registros similares  
✅ **Aplicar** reglas de calidad de datos

---

## 📚 Contenido

1. Validación de CUIT
2. Validación de Email
3. Similitud de Strings (Levenshtein)
4. Deduplicación Fuzzy
5. Consolidación de Registros
6. Caso Integrador

---

## 💡 Por Qué Importa

**Duplicados fuzzy son el enemigo silencioso:**

* "ACME CORP" vs "Acme Corp SA" vs "ACME CORPORATION"
* Mismo cliente, 3 registros
* Facturas dispersas, reportes incorrectos

In [0]:
import pandas as pd
import numpy as np
import re
from difflib import SequenceMatcher

print("🔍 DEPURACIÓN DE MAESTROS")
print("="*70)
print("• Problema: Duplicados fuzzy (similares pero no idénticos)")
print("• Solución: Similitud de strings + reglas de validación")
print("\n📖 Técnicas:")
print("  - Validación con regex")
print("  - Similitud de strings (Levenshtein)")
print("  - Reglas de consolidación")
print("="*70)
print("✅ Librerías cargadas")

In [0]:
import pandas as pd
import re

print("✅ VALIDACIÓN DE CUIT (Argentina)")
print("="*70)

def validar_cuit(cuit):
    """Valida formato CUIT argentino: XX-XXXXXXXX-X (11 dígitos)"""
    # Quitar guiones y espacios
    cuit_limpio = re.sub(r'[^0-9]', '', str(cuit))
    
    # Debe tener 11 dígitos
    if len(cuit_limpio) != 11:
        return False
    
    # Validación de dígito verificador (simplificado)
    # En producción, usar algoritmo completo
    return True

df = pd.DataFrame({
    'Cliente': ['Acme Corp', 'TechStart', 'FinPlus', 'DataCo', 'LogiExp'],
    'CUIT': ['20-12345678-9', '27-987654321', '23-11223344-5', '12345', '30-45678901-2']
})

print("\n📄 Clientes con CUIT:")
print(df)

print("\n" + "="*70)
print("\n✅ VALIDAR FORMATO")
df['CUIT_valido'] = df['CUIT'].apply(validar_cuit)
print(df[['Cliente', 'CUIT', 'CUIT_valido']])

print("\n⚠️  CUITs inválidos:")
invalidos = df[~df['CUIT_valido']]
if len(invalidos) > 0:
    print(invalidos[['Cliente', 'CUIT']])
else:
    print("  Ninguno")

print("\n" + "="*70)
print("✅ Validación completada")

In [0]:
import pandas as pd
import re

print("📧 VALIDACIÓN DE EMAIL")
print("="*70)

def validar_email(email):
    """Valida formato de email con regex"""
    if pd.isna(email):
        return False
    
    patron = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(patron, str(email)))

df = pd.DataFrame({
    'Cliente': ['Acme', 'TechStart', 'FinPlus', 'DataCo', 'LogiExp'],
    'Email': ['contacto@acme.com', 'info@techstart', 'ventas@finplus.com', '@dataco.com', 'soporte@logiexp.com']
})

print("\n📄 Clientes con Email:")
print(df)

print("\n" + "="*70)
print("\n✅ VALIDAR FORMATO")
df['Email_valido'] = df['Email'].apply(validar_email)
print(df)

print("\n⚠️  Emails inválidos:")
invalidos = df[~df['Email_valido']]
if len(invalidos) > 0:
    print(invalidos[['Cliente', 'Email']])
    print(f"\nTotal inválidos: {len(invalidos)} de {len(df)}")

print("\n" + "="*70)
print("✅ Validación completada")

In [0]:
import pandas as pd
from difflib import SequenceMatcher

print("🔍 SIMILITUD DE STRINGS (Fuzzy Matching)")
print("="*70)

def similitud(s1, s2):
    """Calcula similitud entre 2 strings (0.0 a 1.0)"""
    return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()

print("\n1️⃣  EJEMPLOS DE SIMILITUD")
ejemplos = [
    ('ACME CORP', 'Acme Corp SA'),
    ('ACME CORP', 'ACME CORPORATION'),
    ('ACME CORP', 'TechStart SRL'),
    ('TechStart', 'TechStart SRL'),
    ('DataCo', 'Data Co SA')
]

for s1, s2 in ejemplos:
    sim = similitud(s1, s2)
    print(f"  '{s1}' vs '{s2}': {sim:.2%}")

print("\n" + "="*70)
print("\n2️⃣  DETECTAR DUPLICADOS FUZZY")

df = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5],
    'Cliente': ['ACME CORP', 'Acme Corp SA', 'TechStart SRL', 'techstart srl', 'FinPlus SA']
})

print("\n📄 Clientes:")
print(df)

print("\n🔍 Comparar todos con todos (umbral 80%):")
UMBRAL = 0.80
duplicados = []

for i in range(len(df)):
    for j in range(i+1, len(df)):
        sim = similitud(df.loc[i, 'Cliente'], df.loc[j, 'Cliente'])
        if sim >= UMBRAL:
            duplicados.append({
                'ID_1': df.loc[i, 'ID'],
                'Cliente_1': df.loc[i, 'Cliente'],
                'ID_2': df.loc[j, 'ID'],
                'Cliente_2': df.loc[j, 'Cliente'],
                'Similitud': f"{sim:.2%}"
            })

if duplicados:
    df_dup = pd.DataFrame(duplicados)
    print(df_dup)
    print(f"\n⚠️  {len(duplicados)} pares de duplicados detectados")
else:
    print("  Ningún duplicado detectado")

print("\n" + "="*70)
print("✅ Detección fuzzy completada")

In [0]:
import pandas as pd
import re
from difflib import SequenceMatcher

print("💼 CASO INTEGRADOR: DEPURACIÓN COMPLETA DE MAESTRO")
print("="*70)

def validar_cuit(cuit):
    cuit_limpio = re.sub(r'[^0-9]', '', str(cuit))
    return len(cuit_limpio) == 11

def validar_email(email):
    if pd.isna(email):
        return False
    patron = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(patron, str(email)))

def similitud(s1, s2):
    return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()

# Dataset con múltiples problemas
df = pd.DataFrame({
    'ID': [1, 2, 3, 4, 5, 6],
    'Cliente': ['ACME CORP', '  Acme Corp SA  ', 'TechStart SRL', 'techstart srl', 'FinPlus SA', 'DataCo SRL'],
    'CUIT': ['20-12345678-9', '20-12345678-9', '27-98765432-1', '27987654321', '12345', '30-45678901-2'],
    'Email': ['contacto@acme.com', 'ventas@acme.com', 'info@techstart.com', 'info@techstart', 'ventas@finplus.com', 'admin@dataco.com']
})

print("\n📄 DATOS ORIGINALES:")
print(df)

print("\n" + "="*70)
print("\n🔍 PROCESO DE DEPURACIÓN")
print("-"*70)

print("\nPaso 1: Normalizar nombres")
df['Cliente_norm'] = df['Cliente'].str.strip().str.upper()

print("\nPaso 2: Validar CUIT")
df['CUIT_valido'] = df['CUIT'].apply(validar_cuit)
print(f"  CUITs inválidos: {(~df['CUIT_valido']).sum()}")

print("\nPaso 3: Validar Email")
df['Email_valido'] = df['Email'].apply(validar_email)
print(f"  Emails inválidos: {(~df['Email_valido']).sum()}")

print("\nPaso 4: Detectar duplicados fuzzy (umbral 80%)")
UMBRAL = 0.80
duplicados = []

for i in range(len(df)):
    for j in range(i+1, len(df)):
        sim = similitud(df.loc[i, 'Cliente_norm'], df.loc[j, 'Cliente_norm'])
        if sim >= UMBRAL:
            duplicados.append((df.loc[i, 'ID'], df.loc[j, 'ID'], sim))

print(f"  Pares duplicados: {len(duplicados)}")
if duplicados:
    for id1, id2, sim in duplicados:
        c1 = df[df['ID']==id1]['Cliente_norm'].values[0]
        c2 = df[df['ID']==id2]['Cliente_norm'].values[0]
        print(f"    ID {id1} vs {id2}: '{c1}' vs '{c2}' ({sim:.2%})")

print("\n" + "="*70)
print("\n📊 REPORTE DE CALIDAD")
print("-"*70)
print(f"Total registros: {len(df)}")
print(f"CUITs válidos: {df['CUIT_valido'].sum()} ({df['CUIT_valido'].sum()/len(df)*100:.1f}%)")
print(f"Emails válidos: {df['Email_valido'].sum()} ({df['Email_valido'].sum()/len(df)*100:.1f}%)")
print(f"Duplicados fuzzy: {len(duplicados)} pares")

print("\n⚠️  Registros con problemas:")
problemas = df[~df['CUIT_valido'] | ~df['Email_valido']]
if len(problemas) > 0:
    print(problemas[['ID', 'Cliente', 'CUIT_valido', 'Email_valido']])

print("\n" + "="*70)
print("✅ Depuración completada - Revisar duplicados y datos inválidos")

## 🎓 Conclusiones

### ✅ Lo Que Aprendiste

1. **Validación con Regex:**
   - CUIT: 11 dígitos
   - Email: `nombre@dominio.ext`
   - Teléfono: patrones numéricos

2. **Similitud de Strings:**
   - `SequenceMatcher().ratio()`
   - Umbral típico: 80-85%
   - Comparación todos-con-todos

3. **Deduplicación Fuzzy:**
   - Normalizar primero
   - Calcular similitud
   - Consolidar registros similares

---

### 🚀 Próximo Notebook

**04_05 - Tratamiento de Outliers y Formato**
* Detección de outliers (IQR, Z-score)
* Tratamiento (cap, winsorize)
* Formatos de fecha y moneda

---

<div style="background: linear-gradient(90deg, #dc2626 0%, #f87171 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔍 ¡Depuración de Maestros Dominada!</h3>
  <p><i>"Validar y deduplicar datos = Base sólida para análisis."</i></p>
</div>